# 00: Environment check and raw data contract

It reads Parquet **footers only**. No row group is decompressed, so the whole notebook completes in a couple of seconds against 86.6 M rows.

## Setup

Find the repository root by walking upwards, then put it on `sys.path` so both packages import cleanly regardless of where
Jupyter was started.

In [ ]:
# Reload edited .py modules without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "oem_analysis").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from oem_analysis.config import se_config as C
from oem_analysis.lib import se_store
from oem_analysis.lib import se_diagnostics as diag

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Repository root:", REPO_ROOT)
print("Data root:      ", C.DATA_ROOT)

## 1. Environment

Required packages must be present. 

Optional ones are informational.

Paths must resolve. 

In [ ]:
env = diag.environment_report()
display(env)

missing = env.loc[env["required"] & ~env["pass"]]
if len(missing):
    raise RuntimeError(f"Required items missing:\n{missing[['item', 'detail']]}")
print("Environment OK.")

## 2. Conventions

Everything the ingest step will apply to turn telemetry into the common CICCADA convention. 

This dataset seems to use the conventions:

- **Active power** is reported as a production magnitude: Over all of 2025 it has `min = 0` and no negative values, so it is already generator-positive. No change.
- **Reactive power** ASSUMED reported in the **generator convention**, where  *negative = absorbing*. AS/NZS 4777.2 Fig 3.2 use the **generatorconvention**, where *negative = absorbing*. So `Q` is multiplied by `+1` at ingest.


[**BMS Note after running notebook 03**: Some sites showcase adverse response with correct magnitude. Is this a case of inverter logging the sign on a different convention? Or genuine adversity?]

In [ ]:
display(C.describe_conventions())

## 3. Connect

An in-memory DuckDB connection. 

No data is copied into a database file.

The store stays as Parquet on disk, readable by pandas, polars and Arrow, and regenerable from the raw delivery.

`se_raw` spans all 12 monthly files as a single relation. `se_alias` is the site mapping CSV. Store tables are registered only once they have been built.

In [ ]:
con = se_store.connect(verbose=True)
display(se_store.relations(con))

## 4. Raw inventory

Footer metadata for each delivered file.

Two columns matter beyond the row counts:

- **`schema_fingerprint`**: A hash of the ordered `(column, type)` list. One distinct value across all 12 files means one schema, so a single query can read the lot.
- **`n_row_groups`**: every file has exactly **one** row group holding millions of rows. There are therefore no row-group statistics to prune on, and any predicate forces a full column scan.

In [ ]:
inv = diag.raw_inventory(con)
display(inv[["month", "n_rows", "n_row_groups", "rows_per_row_group",
             "n_columns", "size_mb", "schema_fingerprint"]])

print(f"Files:        {len(inv)}")
print(f"Total rows:   {inv.n_rows.sum():,}")
print(f"Total size:   {inv.size_mb.sum():,.0f} MB compressed on disk")
print(f"Writer:       {inv.created_by.iloc[0]}")
print()
print("For scale: as float64 in pandas this would be roughly "
      f"{inv.n_rows.sum() * 14 * 8 / 1024**3:.1f} GB in memory, "
      "which is why nothing is ever loaded whole.")

## 5. Raw schema

The 15 delivered columns. `se_config.RAW_COLUMNS` is the declared contract.

**absent**: no nameplate capacity, no inverter model, no DNSP, no install date, and no irradiance. 

Site metadata is postcode and state only.

In [ ]:
schema = diag.raw_schema(con, Path(inv.path.iloc[0]))
schema["declared"] = schema.column_name.map(C.RAW_COLUMNS)
display(schema)

## 6. D1 checks

Three groups of assertions:

1. **Schema contract**: all files share one schema, and it is the declared one.
2. **Inventory**: 12 months with no gaps, and per-file row counts matching `se_config.EXPECTED_RAW_ROWS` (measured 12 Aug 2026).
3. **Alias mapping**: complete, unique, and every state has a timezone mapped.

In [ ]:
checks = diag.run_d1_checks(con, inventory=inv)
display(checks)

ok = diag.summarise(checks, label="D1")
assert ok, "D1 checks failed. see above before proceeding to D2."

## 7. Site mapping

The complete site metadata: 1,602 aliases, each with a postcode and a state.

The 1,602 sites resolve to a few hundred distinct postcodes, so the BOM irradiance extract in (which is keyed on the nearest BOM
grid point, not on the site) will be far smaller than the site count suggests.

In [ ]:
display(diag.alias_mapping_summary(con))

postcodes = se_store.q(con, "SELECT count(DISTINCT zip_code) AS n_postcodes FROM se_alias")
print(f"Distinct postcodes across the fleet: {int(postcodes.n_postcodes.iloc[0])}")
print("These collapse further into BOM grid points at D4, which sets the D12a extract size.")

## 8. Store status

What has been built so far. Everything is expected to be missing at this point.

This table is the SolarEdge analogue of `conformance_queries.table_provenance()`: it
makes it impossible to run an analysis against a store you believed was complete but
was not.

In [ ]:
display(se_store.store_status(con)[["logical_name", "exists", "kind", "size_mb", "n_rows"]])